# Orientation Bug Replication

Creates synthetic ellipse polygons at **known** angles and runs them through both orientation paths:

1. **`fitEllipse`** (scikit-image `EllipseModel`) → returns `theta` in radians
2. **`boulder_row`** (Shapely minimum rotated rectangle → `azimuth()`) → returns `angle`/`angle180`

For each input angle we compare what the code returns vs. what it should return.
If a column is constant regardless of input → that path contains the bug.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from shapely import segmentize
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm

from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row, azimuth
from rastertools_BOULDERING import metadata as raster_metadata

## Helper: generate a synthetic ellipse polygon at a known angle

In [ ]:
def make_ellipse_polygon(a, b, theta_degrees, cx=0.0, cy=0.0, n_pts=200):
    theta_rad = np.radians(theta_degrees)
    t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    x = a * np.cos(t)
    y = b * np.sin(t)
    x_rot = x * np.cos(theta_rad) - y * np.sin(theta_rad) + cx
    y_rot = x * np.sin(theta_rad) + y * np.cos(theta_rad) + cy
    return Polygon(np.column_stack([x_rot, y_rot]))


def make_noisy_ellipse_polygon(a, b, theta_degrees, cx=0.0, cy=0.0, n_pts=40, noise_scale=0.05):
    rng = np.random.default_rng(seed=42)
    theta_rad = np.radians(theta_degrees)
    t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    noise = rng.normal(0, noise_scale * b, size=n_pts)
    r = 1.0 + noise
    x = a * np.cos(t) * r
    y = b * np.sin(t) * r
    x_rot = x * np.cos(theta_rad) - y * np.sin(theta_rad) + cx
    y_rot = x * np.sin(theta_rad) + y * np.cos(theta_rad) + cy
    return Polygon(np.column_stack([x_rot, y_rot]))

In [ ]:
A   = 15.0
B   = 10.0
CX  = 500.0
CY  = 500.0
RES = 1.0

test_angles = [0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165]

---
## Part 1: Raw paths (independent)

In [ ]:
results = []
for deg in test_angles:
    poly = make_ellipse_polygon(A, B, deg, cx=CX, cy=CY)
    row  = pd.Series({"geometry": poly})
    try:
        _, a_fit, b_fit, theta_rad = fitEllipse(row)
        theta_deg = np.degrees(theta_rad)
    except Exception as e:
        theta_deg, a_fit, b_fit = None, None, None
        print(f"fitEllipse failed at {deg}°: {e}")
    try:
        mrr_row = pd.Series({"geometry": poly.minimum_rotated_rectangle})
        (_, _, _, _, _, angle_default, angle360, angle180) = boulder_row(mrr_row)
    except Exception as e:
        angle_default, angle360, angle180 = None, None, None
        print(f"boulder_row failed at {deg}°: {e}")
    results.append({"input_deg": deg, "fitEllipse_theta_deg": theta_deg,
                    "fitEllipse_a": a_fit, "fitEllipse_b": b_fit,
                    "mrr_angle_default": angle_default, "mrr_angle180": angle180})

df = pd.DataFrame(results)
df

In [ ]:
expected = [d % 180 for d in test_angles]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(test_angles, expected, 'k--', label='expected (input mod 180)')
axes[0].plot(test_angles, df["fitEllipse_theta_deg"] % 180, 'ro-', label='fitEllipse theta')
axes[0].set_title("Path 1: fitEllipse (EllipseModel)"); axes[0].legend(); axes[0].grid(True)
axes[1].plot(test_angles, expected, 'k--', label='expected (input mod 180)')
axes[1].plot(test_angles, df["mrr_angle180"], 'bs-', label='MRR angle180')
axes[1].set_title("Path 2: boulder_row (MRR + azimuth)"); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.savefig("orientation_part1_raw_paths.png", dpi=150); plt.show()

---
## Part 2: Full post-processing pipeline (chained)

```
raw boulder polygon → segmentize → fitEllipse → MRR → boulder_row → final angle
```

In [ ]:
def full_pipeline(poly, res=RES):
    poly_seg = segmentize(poly, res)
    row_seg  = pd.Series({"geometry": poly_seg})
    ellipse_poly, a_fit, b_fit, theta_rad = fitEllipse(row_seg)
    theta_deg = np.degrees(theta_rad)
    mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
    (_, _, long_axis, short_axis, _, angle_default, angle360, angle180) = boulder_row(mrr_row)
    return {"fitEllipse_theta_deg": theta_deg, "fitEllipse_a": a_fit, "fitEllipse_b": b_fit,
            "final_angle_default": angle_default, "final_angle180": angle180,
            "aspect_ratio": long_axis / short_axis if short_axis > 0 else None}

In [ ]:
results_clean, results_noisy = [], []
for deg in test_angles:
    for results_list, poly_fn in [(results_clean, make_ellipse_polygon),
                                   (results_noisy, make_noisy_ellipse_polygon)]:
        poly = poly_fn(A, B, deg, cx=CX, cy=CY)
        try:
            r = full_pipeline(poly)
        except Exception as e:
            print(f"Pipeline failed at {deg}°: {e}")
            r = {k: None for k in ["fitEllipse_theta_deg","fitEllipse_a","fitEllipse_b",
                                    "final_angle_default","final_angle180","aspect_ratio"]}
        results_list.append({"input_deg": deg, **r})

df_clean = pd.DataFrame(results_clean)
df_noisy = pd.DataFrame(results_noisy)
print("=== Clean ==="); print(df_clean.to_string())
print("\n=== Noisy ==="); print(df_noisy.to_string())

In [ ]:
expected = [d % 180 for d in test_angles]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for row_i, (df_i, label) in enumerate([(df_clean, "Clean"), (df_noisy, "Noisy")]):
    for col_i, (col, title) in enumerate([("fitEllipse_theta_deg", "fitEllipse theta"),
                                           ("final_angle180", "final angle180")]):
        axes[row_i, col_i].plot(test_angles, expected, 'k--', label='expected')
        axes[row_i, col_i].plot(test_angles, df_i[col] % 180 if col == "fitEllipse_theta_deg" else df_i[col],
                                 'ro-' if col_i == 0 else 'bs-', label=title)
        axes[row_i, col_i].set_title(f"{label} input: {title}")
        axes[row_i, col_i].legend(); axes[row_i, col_i].grid(True)
plt.tight_layout(); plt.savefig("orientation_part2_full_pipeline.png", dpi=150); plt.show()

---
## Part 3: Real boulder data

Uses the output of `example_of_tiled_predictions_fast.ipynb` (the `inference_fast3` shapefiles).
Run that notebook first to generate the detection shapefiles.

This part:
1. Loads real boulder mask polygons
2. Runs the full orientation pipeline on them
3. Checks how many ellipse fits fail (theta fallback = 1.0 rad)
4. Plots the orientation distribution

In [ ]:
# --- Paths (must match example_of_tiled_predictions_fast.ipynb) ---
home_p    = Path.home() / "Downloads" / "Test_ManualBoulderNetEnvironment"
work_dir  = home_p / "tmp" / "YOLOv8BeyondEarth"
raster_dir = work_dir / "raster"
output_dir = work_dir / "inference_fast3"
in_raster  = raster_dir / "M139694087LE.tif"

# Resolution from the raster (needed for filter_by_area and segmentize)
res = raster_metadata.get_resolution(in_raster)[0]
areal_threshold = (res * res) * (4.74 * 4.74)
print(f"Resolution: {res:.4f} m, areal threshold: {areal_threshold:.4f} m²")

# Find all NMS mask shapefiles produced by the detection notebook
mask_shps = sorted(output_dir.glob("*-downscaled-mask-nms.shp"))
print(f"Found {len(mask_shps)} mask shapefiles:")
for p in mask_shps:
    print(" ", p.name)

In [ ]:
# Load and merge all mask shapefiles, then filter by area
gdfs = [gpd.read_file(p) for p in mask_shps]
gdf_all = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
gdf_all["poly_area"] = gdf_all.geometry.area
gdf_filtered = gdf_all[gdf_all["poly_area"] >= areal_threshold].reset_index(drop=True)
print(f"Total detections: {len(gdf_all)}, after area filter: {len(gdf_filtered)}")

In [ ]:
# Run the full orientation pipeline on each real boulder polygon
# Tracks how many ellipse fits fail (fallback theta=1.0)
fit_failures = 0
records = []

for idx, row in tqdm(gdf_filtered.iterrows(), total=len(gdf_filtered)):
    poly = row.geometry
    poly_seg = segmentize(poly, res)
    row_seg  = pd.Series({"geometry": poly_seg})

    # fitEllipse
    try:
        ellipse_poly, a_fit, b_fit, theta_rad = fitEllipse(row_seg)
        fit_failed = False
    except Exception:
        ellipse_poly = poly
        a_fit, b_fit, theta_rad = 1.0, 1.0, 1.0
        fit_failed = True
        fit_failures += 1

    theta_deg = np.degrees(theta_rad)

    # MRR → boulder_row
    try:
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        (_, _, long_axis, short_axis, _, angle_default, angle360, angle180) = boulder_row(mrr_row)
        aspect_ratio = long_axis / short_axis if short_axis > 0 else None
    except Exception:
        angle_default, angle360, angle180, aspect_ratio = None, None, None, None

    records.append({
        "theta_deg"    : theta_deg,
        "fit_failed"   : fit_failed,
        "angle180"     : angle180,
        "aspect_ratio" : aspect_ratio,
    })

df_real = pd.DataFrame(records)
print(f"\nFit failures: {fit_failures} / {len(gdf_filtered)} ({100*fit_failures/len(gdf_filtered):.1f}%)")
df_real.describe()

In [ ]:
# Filter to boulders with aspect ratio 1.2–2.0 (the range of interest)
df_elongated = df_real[(df_real["aspect_ratio"] >= 1.2) & (df_real["aspect_ratio"] <= 2.0)]
print(f"Elongated boulders (1.2 ≤ a/b ≤ 2.0): {len(df_elongated)} / {len(df_real)}")
print(f"Of those, fit failures: {df_elongated['fit_failed'].sum()}")

# Check if theta=1.0 rad (57.3°) is overrepresented — the fallback value
print(f"\ntheta_deg value counts (top 10):")
print(df_elongated["theta_deg"].round(1).value_counts().head(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution of theta (from fitEllipse)
axes[0].hist(df_elongated["theta_deg"].dropna(), bins=36, range=(-180, 180), color='tomato', edgecolor='k')
axes[0].axvline(np.degrees(1.0), color='navy', linestyle='--', label=f'fallback = {np.degrees(1.0):.1f}°')
axes[0].set_title("fitEllipse theta (elongated boulders)")
axes[0].set_xlabel("theta (°)"); axes[0].legend()

# Distribution of final angle180 (from boulder_row)
axes[1].hist(df_elongated["angle180"].dropna(), bins=36, range=(0, 180), color='steelblue', edgecolor='k')
axes[1].set_title("Final angle180 (elongated boulders)")
axes[1].set_xlabel("angle180 (°)")

# aspect ratio distribution
axes[2].hist(df_real["aspect_ratio"].dropna(), bins=40, color='seagreen', edgecolor='k')
axes[2].axvline(1.2, color='k', linestyle='--', label='1.2')
axes[2].axvline(2.0, color='k', linestyle=':', label='2.0')
axes[2].set_title("Aspect ratio distribution (all boulders)")
axes[2].set_xlabel("a/b"); axes[2].legend()

plt.tight_layout()
plt.savefig("orientation_part3_real_data.png", dpi=150)
plt.show()
print("Plot saved to orientation_part3_real_data.png")

### Part 3 interpretation

- **`theta_deg` histogram has a spike at 57.3°** → many fits failing, fallback `theta=1.0` is the bug
- **`angle180` histogram is flat/uniform** → orientations are actually fine, the issue is in which column is being read
- **`angle180` histogram has a spike at one value** → `boulder_row`/`azimuth` is broken on real polygon shapes
- **Both histograms look reasonable (spread across 0–180°)** → the bug is further downstream in how results are used or visualized

---
## Part 4: Size threshold sweep

Tests the hypothesis that the spikes at ~90° and ~180° are caused by small boulders with
blocky pixel-aligned mask outlines. Larger boulders have more pixels, so the staircase
artifact is a smaller fraction of their perimeter and the ellipse fit sees more of the true shape.

If the spikes shrink as the minimum area threshold increases, the pixel-grid artifact is confirmed.

**Note:** `df_real` and `gdf_filtered` must be in scope from running Part 3.

In [ ]:
# Add poly_area back into df_real so we can filter by size
df_real["poly_area"] = gdf_filtered["poly_area"].values

# Size thresholds to sweep (m²)
size_thresholds = [
    (areal_threshold, f"all (>{areal_threshold:.1f} m²)"),
    (5.0,   ">5 m²"),
    (10.0,  ">10 m²"),
    (25.0,  ">25 m²"),
    (50.0,  ">50 m²"),
    (100.0, ">100 m²"),
]

print(f"{'Threshold':>20}  {'n elongated':>12}  {'spike ~90°':>12}  {'spike ~180°':>12}")
print("-" * 62)
for min_area, label in size_thresholds:
    subset = df_real[
        (df_real["poly_area"] >= min_area) &
        (df_real["aspect_ratio"] >= 1.2) &
        (df_real["aspect_ratio"] <= 2.0)
    ]
    n = len(subset)
    spike_90  = ((subset["theta_deg"] > 80)  & (subset["theta_deg"] < 100)).sum()
    spike_180 = ((subset["theta_deg"] > 165) | (subset["theta_deg"] < -165)).sum()
    print(f"{label:>20}  {n:>12}  {spike_90:>12}  {spike_180:>12}")

In [ ]:
n_thresholds = len(size_thresholds)
fig, axes = plt.subplots(2, n_thresholds, figsize=(4 * n_thresholds, 8))

for col, (min_area, label) in enumerate(size_thresholds):
    subset = df_real[
        (df_real["poly_area"] >= min_area) &
        (df_real["aspect_ratio"] >= 1.2) &
        (df_real["aspect_ratio"] <= 2.0)
    ]

    axes[0, col].hist(subset["theta_deg"].dropna(), bins=36, range=(0, 180),
                      color='tomato', edgecolor='k')
    axes[0, col].set_title(f"theta\n{label}\n(n={len(subset)})")
    axes[0, col].set_xlabel("theta (°)")
    if col == 0:
        axes[0, col].set_ylabel("count")

    axes[1, col].hist(subset["angle180"].dropna(), bins=36, range=(0, 180),
                      color='steelblue', edgecolor='k')
    axes[1, col].set_title(f"angle180\n{label}\n(n={len(subset)})")
    axes[1, col].set_xlabel("angle180 (°)")
    if col == 0:
        axes[1, col].set_ylabel("count")

plt.suptitle(
    "Orientation distribution by minimum boulder size\n"
    "Spikes at 90° and 180° should shrink as size increases if pixel-grid artifact is the cause",
    fontsize=12)
plt.tight_layout()
plt.savefig("orientation_part4_size_sweep.png", dpi=150)
plt.show()
print("Plot saved to orientation_part4_size_sweep.png")

### Part 4 interpretation

- **Spikes at 90°/180° shrink as threshold increases** → confirms the pixel-grid artifact hypothesis. Small boulders with boxy masks are the source; larger boulders have smoother outlines and more physically meaningful orientations.
- **Spikes persist even at large sizes** → the artifact is not purely a size issue — mask quality is poor even for larger boulders, likely because `downscale_pred=True` reduces effective pixel resolution for all sizes.
- **Distribution becomes more uniform at large sizes** → orientation is only reliable for larger boulders with this pipeline. SAM2 masks would extend reliable orientation estimation down to smaller boulders.